In [3]:
import duckdb
import pandas as pd
from pathlib import Path
from diskcache import Cache
from itertools import chain
from collections import Counter
import contextlib
from unidecode import unidecode
from nameparser import HumanName

from dataclasses import dataclass
from itertools import chain

import igraph as ig
import matplotlib.pyplot as plt
import seaborn.objects as so
import seaborn as sns

from utils.pandas_setup import pandas_setup
pandas_setup()

import pyalex
from pyalex import Works, Authors, Sources, Institutions, Topics, Publishers, Funders
pyalex.config.email = "Lawrence.Cram@anu.edu.au"
pyalex.config.max_retries = 0
pyalex.config.retry_backoff_factor = 0.1
pyalex.config.retry_http_codes = [429, 500, 503]



def normalise_name(in_name: str=None) -> list:
    # print(f'{in_name = }')
    in_name = ' '.join([part.strip() for part in unidecode(in_name).split(' ')])
    in_name = in_name.title()
    name = HumanName(in_name)
    
    if name.last == 'Athey':
        name.first = 'Susan.'
    if name.last == 'Deaton':
        name.first = 'Angas.'

    if name.middle == "":
        fullname = f'{name.first} {name.last}'
    else:
        fullname = f'{name.first} {name.middle} {name.last}'

    if fullname == 'David A Hensher':
        fullname = 'David A. Hensher'
    if fullname == 'Ja Robinson':
        fullname = 'James A. Robinson'
    if fullname == "Nicola Fuchs-Schuendeln":
        fullname = "Nicola Fuchs-Schundeln"
    if fullname == "Georg Weizsaecker":
        fullname = "Georg Weizsacker"
    if fullname == "Che Yeon-Koo":
        fullname = "Yeon-Koo Che"
    if fullname == "Dong-Yang Zhang":
        fullname = "Dongyang Zhang"

    # return {'first': name.first, 'middle': name.middle, 'family': name.last, 'fullname': fullname}
    return [name.first, name.middle, name.last, fullname]


def dummy(input: str=None) -> list:
    return {'key1': 1, 'key2': 2}



In [ ]:
MY_DATA_PATH = Path('../DATA/')
MY_DATABASE_FILE = Path(MY_DATA_PATH / 'econ.duckdb')
MY_CACHE_FILE = Path('/home/lc/m/.cache/economicsbusiness/cache.db')
DATAFILES_PATH = Path('../DATAFILES')

class SetUp:

    def __init__(self):
        self._open_db()
        self._open_cache()
        return

    def _open_db(self):
        self.db = duckdb.connect()
        self.db.sql(f"ATTACH IF NOT EXISTS '{str(MY_DATA_PATH)}' AS funder")
        # self.db.sql("ATTACH IF NOT EXISTS '/home/lc/m/openalex_dec24/duckdb/authors.duckdb'")
        # self.db.sql("ATTACH IF NOT EXISTS '/home/lc/m/openalex_dec24/duckdb/institutions.duckdb'")
        # self.db.sql("ATTACH IF NOT EXISTS '/home/lc/m/working/orcid.duckdb'")
        for table in ['subfields2groups', 'old_for_to_new_for']:
            self.db.sql(f"DROP TA
        self._open_db()  null_handling='special',
                                    side_effects=True
                                )
        print(self.db.sql("SHOW ALL TABLES").df())
        return
    
    def _open_cache(self):
        self.cache = Cache(f'{str(CACHE_PATH)}', size_limit=int(4e9))
        # self.cache.clear()
        return
    
    def _cache_manager(self, task=None):
        # Check cache
        result = self.cache.get(task)
        if isinstance(result, pd.DataFrame):
            print("\rC", end='\r')
            return result
        if isinstance(result, list):
            print("\rC", end='\r')
            self.cache[task] = pd.json_normalize(result)
            return self.cache[task]
        if result == "__NONE__":
            print("\rC", end='\r')
            return None

        # Not in cache, go to Web
        result = self._extractor(task=task)
        if isinstance(result, (list, str)):
            print("\rW", end='\r')
            self.cache[task] = pd.json_normalize(result)
            return self.cache[task]
        if result is None:
            print("\rW", end='\r')
            self.cache[task] = "__NONE__"
            return None
        return None
        
    def _extractor(self, task=None):
        # print(f'_extractor - {task = }')
        response = eval(task)
        # print(f'{type(response) = } {response.text = }')
        return response.text.replace("'", "''")
    
    def name_of_global_obj(self, obj=None):
        for objname, oid in globals().items():
            if oid is obj:
                return objname

In [5]:
s = SetUp()

NameError: name 'FUNDER_DATA_PATH' is not defined